# Logit vs Probit: Comparing Link Functions for A/B Testing

This notebook compares **logit** and **probit** link functions for analyzing A/B tests.

## When to use which?

- **Logit**: Industry standard, interpretable odds ratios, robust to outliers
- **Probit**: Assumes normal latent variable, sometimes better for certain data patterns

Both usually give similar results for treatment effects on the probability scale.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from ab_glm import (
    simulate_ab_data,
    fit_binomial_glm,
    marginal_effects_ate_and_rr,
    brier_score
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Understanding Logit and Probit Link Functions

Both link functions map linear predictors to probabilities [0,1], but with different shapes.

In [ ]:
# Visualize the link functions
x = np.linspace(-4, 4, 100)
logit_probs = 1 / (1 + np.exp(-x))
probit_probs = stats.norm.cdf(x)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Link functions
axes[0].plot(x, logit_probs, label='Logit', linewidth=2, color='#e74c3c')
axes[0].plot(x, probit_probs, label='Probit', linewidth=2, color='#3498db')
axes[0].set_xlabel('Linear Predictor (Xβ)')
axes[0].set_ylabel('Probability')
axes[0].set_title('Link Functions: Linear Predictor → Probability')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)

# Difference between functions
diff = logit_probs - probit_probs
axes[1].plot(x, diff * 100, linewidth=2, color='#9b59b6')
axes[1].set_xlabel('Linear Predictor (Xβ)')
axes[1].set_ylabel('Difference (percentage points)')
axes[1].set_title('Logit - Probit Difference')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='gray', linestyle='-', alpha=0.5)
axes[1].fill_between(x, 0, diff * 100, alpha=0.3, color='#9b59b6')

plt.tight_layout()
plt.show()

print("Key differences:")
print("• Logit has slightly heavier tails (more probability mass at extremes)")
print("• Probit is based on the normal CDF")
print("• Maximum difference occurs around ±1.6 on the linear scale")
print("• Both are symmetric around 0.5 probability")

## 2. Generate Test Data

In [ ]:
# Generate data with known treatment effect
np.random.seed(42)
df = simulate_ab_data(n_users=3000, sessions_per_user=(2, 6), seed=42)

print(f"Dataset: {df.shape[0]:,} sessions from {df['user_id'].nunique():,} users")
print(f"Treatment rate: {df.groupby('user_id')['T'].first().mean():.2%}")
print(f"Overall conversion rate: {df['y'].mean():.2%}")

## 3. Fit Both Models

In [ ]:
# Fit logit model
print("Fitting LOGIT model...")
glm_logit, _, df_logit, results_logit = fit_binomial_glm(df, link="logit")
ate_logit, rr_logit, p_treat_logit, p_ctrl_logit = marginal_effects_ate_and_rr(
    results_logit, df_logit
)

# Fit probit model
print("Fitting PROBIT model...")
glm_probit, _, df_probit, results_probit = fit_binomial_glm(df, link="probit")
ate_probit, rr_probit, p_treat_probit, p_ctrl_probit = marginal_effects_ate_and_rr(
    results_probit, df_probit
)

print("\n" + "="*60)
print("MODEL COMPARISON: COEFFICIENT ESTIMATES")
print("="*60)

# Create comparison table
coef_comparison = pd.DataFrame({
    'Logit Coef': results_logit.params,
    'Logit SE': np.sqrt(np.diag(results_logit.cov_params())),
    'Probit Coef': results_probit.params,
    'Probit SE': np.sqrt(np.diag(results_probit.cov_params())),
})

# Add ratio column
coef_comparison['Ratio (Logit/Probit)'] = (
    coef_comparison['Logit Coef'] / coef_comparison['Probit Coef']
)

print(coef_comparison.round(4))
print("\nNote: Logit coefficients are typically ~1.6x larger than probit")
print("This is expected due to the different variance of underlying distributions")

## 4. Compare Treatment Effects on Probability Scale

In [ ]:
# Create comparison of business metrics
comparison = pd.DataFrame({
    'Metric': [
        'Control Probability',
        'Treatment Probability', 
        'ATE (Risk Difference)',
        'Risk Ratio',
        'Relative Lift (%)',
        'Treatment Coef (link scale)',
        'Treatment SE (link scale)'
    ],
    'Logit': [
        p_ctrl_logit,
        p_treat_logit,
        ate_logit,
        rr_logit,
        (rr_logit - 1) * 100,
        results_logit.params['T'],
        np.sqrt(np.diag(results_logit.cov_params()))[list(results_logit.params.index).index('T')]
    ],
    'Probit': [
        p_ctrl_probit,
        p_treat_probit,
        ate_probit,
        rr_probit,
        (rr_probit - 1) * 100,
        results_probit.params['T'],
        np.sqrt(np.diag(results_probit.cov_params()))[list(results_probit.params.index).index('T')]
    ]
})

comparison['Difference'] = comparison['Logit'] - comparison['Probit']
comparison['% Difference'] = (
    (comparison['Logit'] - comparison['Probit']) / comparison['Probit'] * 100
)

print("="*60)
print("TREATMENT EFFECT COMPARISON (PROBABILITY SCALE)")
print("="*60)
print(comparison.round(4).to_string(index=False))

print("\n" + "="*60)
print("KEY INSIGHT")
print("="*60)
print("Despite different coefficient scales, both models produce nearly")
print("IDENTICAL estimates on the probability scale (ATE, RR).")
print("This is why the choice between logit and probit rarely matters in practice.")

## 5. Model Fit Comparison

In [ ]:
# Calculate fit statistics
y_true = df_logit['y'].values
pred_logit = results_logit.predict(df_logit)
pred_probit = results_probit.predict(df_probit)

brier_logit = brier_score(y_true, pred_logit)
brier_probit = brier_score(y_true, pred_probit)

# Log-likelihood
ll_logit = results_logit.llf
ll_probit = results_probit.llf

# AIC
aic_logit = results_logit.aic
aic_probit = results_probit.aic

# BIC  
bic_logit = results_logit.bic
bic_probit = results_probit.bic

print("="*60)
print("MODEL FIT COMPARISON")
print("="*60)

fit_comparison = pd.DataFrame({
    'Metric': ['Brier Score', 'Log-Likelihood', 'AIC', 'BIC'],
    'Logit': [brier_logit, ll_logit, aic_logit, bic_logit],
    'Probit': [brier_probit, ll_probit, aic_probit, bic_probit],
    'Better Model': [
        'Logit' if brier_logit < brier_probit else 'Probit',
        'Logit' if ll_logit > ll_probit else 'Probit',
        'Logit' if aic_logit < aic_probit else 'Probit',
        'Logit' if bic_logit < bic_probit else 'Probit'
    ]
})

print(fit_comparison.round(4).to_string(index=False))
print("\nNote: Lower is better for Brier Score, AIC, and BIC")
print("      Higher is better for Log-Likelihood")

## 6. Prediction Distribution Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Distribution of predictions
axes[0].hist(pred_logit, bins=30, alpha=0.5, label='Logit', color='#e74c3c', density=True)
axes[0].hist(pred_probit, bins=30, alpha=0.5, label='Probit', color='#3498db', density=True)
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution of Predictions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter plot comparing predictions
axes[1].scatter(pred_logit, pred_probit, alpha=0.1, s=1)
axes[1].plot([0, 1], [0, 1], 'r--', alpha=0.5)
axes[1].set_xlabel('Logit Predictions')
axes[1].set_ylabel('Probit Predictions')
axes[1].set_title('Prediction Agreement')
axes[1].grid(True, alpha=0.3)

# Correlation
corr = np.corrcoef(pred_logit, pred_probit)[0, 1]
axes[1].text(0.05, 0.95, f'Correlation: {corr:.4f}', 
            transform=axes[1].transAxes, fontsize=12,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Difference in predictions
diff = pred_logit - pred_probit
axes[2].hist(diff * 100, bins=30, color='#9b59b6', edgecolor='black', alpha=0.7)
axes[2].set_xlabel('Prediction Difference (Logit - Probit) in pp')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Distribution of Prediction Differences')
axes[2].axvline(x=0, color='red', linestyle='--', alpha=0.5)
axes[2].grid(True, alpha=0.3)

# Add statistics
axes[2].text(0.98, 0.98, 
            f'Mean: {np.mean(diff)*100:.3f} pp\n'
            f'Std: {np.std(diff)*100:.3f} pp\n'
            f'Max: {np.max(np.abs(diff))*100:.3f} pp',
            transform=axes[2].transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
            ha='right', va='top')

plt.tight_layout()
plt.show()

print(f"Prediction correlation: {corr:.4f}")
print(f"Mean absolute difference: {np.mean(np.abs(diff))*100:.3f} percentage points")
print(f"Max absolute difference: {np.max(np.abs(diff))*100:.3f} percentage points")

## 7. Robustness Check: Multiple Random Seeds

In [ ]:
# Run multiple simulations to check consistency
n_simulations = 20
results = []

for seed in range(n_simulations):
    # Generate new data
    df_sim = simulate_ab_data(n_users=1000, sessions_per_user=(2, 5), seed=seed)
    
    # Fit both models
    _, _, df_l, res_l = fit_binomial_glm(df_sim, link="logit")
    ate_l, rr_l, _, _ = marginal_effects_ate_and_rr(res_l, df_l)
    
    _, _, df_p, res_p = fit_binomial_glm(df_sim, link="probit")
    ate_p, rr_p, _, _ = marginal_effects_ate_and_rr(res_p, df_p)
    
    results.append({
        'seed': seed,
        'ate_logit': ate_l,
        'ate_probit': ate_p,
        'rr_logit': rr_l,
        'rr_probit': rr_p,
        'ate_diff': ate_l - ate_p,
        'rr_diff': rr_l - rr_p
    })

results_df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ATE comparison
axes[0].scatter(results_df['ate_logit'] * 100, 
               results_df['ate_probit'] * 100, alpha=0.7)
axes[0].plot([0, 10], [0, 10], 'r--', alpha=0.5)
axes[0].set_xlabel('Logit ATE (percentage points)')
axes[0].set_ylabel('Probit ATE (percentage points)')
axes[0].set_title(f'ATE Estimates Across {n_simulations} Simulations')
axes[0].grid(True, alpha=0.3)

# RR comparison
axes[1].scatter(results_df['rr_logit'], results_df['rr_probit'], alpha=0.7)
min_rr = min(results_df['rr_logit'].min(), results_df['rr_probit'].min())
max_rr = max(results_df['rr_logit'].max(), results_df['rr_probit'].max())
axes[1].plot([min_rr, max_rr], [min_rr, max_rr], 'r--', alpha=0.5)
axes[1].set_xlabel('Logit Risk Ratio')
axes[1].set_ylabel('Probit Risk Ratio')
axes[1].set_title(f'Risk Ratio Estimates Across {n_simulations} Simulations')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("="*60)
print(f"CONSISTENCY ACROSS {n_simulations} SIMULATIONS")
print("="*60)
print(f"Mean ATE difference (Logit - Probit): {results_df['ate_diff'].mean()*100:.4f} pp")
print(f"Std of ATE difference: {results_df['ate_diff'].std()*100:.4f} pp")
print(f"\nMean RR difference (Logit - Probit): {results_df['rr_diff'].mean():.4f}")
print(f"Std of RR difference: {results_df['rr_diff'].std():.4f}")
print(f"\nCorrelation of ATE estimates: {results_df['ate_logit'].corr(results_df['ate_probit']):.4f}")
print(f"Correlation of RR estimates: {results_df['rr_logit'].corr(results_df['rr_probit']):.4f}")

## 8. Practical Recommendations

Based on our comprehensive comparison:

In [ ]:
print("="*60)
print("PRACTICAL RECOMMENDATIONS")
print("="*60)
print()
print("1. DEFAULT CHOICE: LOGIT")
print("   • Industry standard in tech and business")
print("   • Coefficients interpretable as log-odds ratios")
print("   • Slightly more robust to outliers (heavier tails)")
print("   • Better documented and understood")
print()
print("2. WHEN TO CONSIDER PROBIT:")
print("   • Academic settings (economics, social sciences)")
print("   • When you believe in normal latent variable model")
print("   • Slightly better fit for some datasets")
print()
print("3. KEY FINDING:")
print("   • Differences in ATE/RR are typically < 0.1 percentage points")
print("   • Both give nearly identical business conclusions")
print("   • Choice rarely affects decision-making")
print()
print("4. ROBUSTNESS CHECK:")
print("   • Run both as sensitivity analysis for important decisions")
print("   • If results differ substantially, investigate why")
print("   • Large differences may indicate model misspecification")
print()
print("5. REPORTING:")
print("   • Always report marginal effects (ATE, RR) not just coefficients")
print("   • These are comparable across link functions")
print("   • Business stakeholders care about probability scale, not link scale")

## Summary

This comparison demonstrates that:

1. **Logit and probit produce nearly identical results** on the probability scale
2. **Coefficient magnitudes differ** (~1.6x) but this is expected and doesn't affect inference
3. **Model fit is usually very similar** between the two approaches
4. **The choice rarely matters** for practical A/B testing decisions

For most A/B testing applications, stick with logit as the industry standard unless you have specific reasons to use probit.